# Data quality and validation splits

Audits the fixed peptide assignments and summarizes the three validation designs.

**Scope:** consumes registered artifacts; no model refitting is performed in this publication notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'publication' else Path.cwd().resolve()
if not (PROJECT / '05_results').exists():
    PROJECT = Path('outputs/CBAC-D-26-03155_revision/06_canonical_revision_project').resolve()
PRIOR = PROJECT.parent / '04_reproducible_analysis/artifacts'
PUB = PROJECT / '05_results/validated/publication_outputs'
TABLES, FIGURES = PUB / 'tables', PUB / 'figures'
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

def read_csv(path, required):
    assert path.exists(), f'Missing registered artifact: {path}'
    df = pd.read_csv(path)
    missing = set(required) - set(df.columns)
    assert not missing, f'{path.name}: missing columns {sorted(missing)}'
    return df

plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 300, 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})
SPLIT_ORDER = ['exact', 'cluster', 'temporal']


In [2]:
splits = read_csv(PROJECT/'02_data/splits/peptide_split_assignments_v1.csv', ['peptide','label_hiconf','exact_split','cluster_split','temporal_split'])
split_cols = [c for c in splits.columns if c.endswith('_split')]
print(f'Rows: {len(splits):,}; unique peptides: {splits.peptide.nunique():,}; positive rate: {splits.label_hiconf.mean():.3f}')
print('Candidate split columns:', split_cols)
display(splits.head())

Rows: 9,672; unique peptides: 9,672; positive rate: 0.092
Candidate split columns: ['exact_split', 'cluster_split', 'temporal_split']


,peptide,total_tested,total_positive,length,pos_rate,n_unique_labels,label_hiconf,earliest_pub_year,latest_pub_year,n_references,pmids,valid_sequence,valid_year,exact_split,edit_distance_2_cluster,cluster_split,temporal_split
0,LSYYKLGASQRVAGD,258.0,181.0,15,0.70155,1,1,2021.0,2024.0,6.0,32999467;33723016;33911008;34529740;35389886;3...,True,2021.0,train,0,test,test
1,RQKKQQTVTLLPAADLDD,228.0,0.0,18,0.00000,1,0,2025.0,2025.0,1.0,41051404,True,2025.0,train,1,train,test
2,TQAFGRRGPEQTQGNFGD,228.0,0.0,18,0.00000,1,0,2025.0,2025.0,1.0,41051404,True,2025.0,train,2,test,test
3,SPDDQIGYYRRATRRIRG,228.0,0.0,18,0.00000,1,0,2025.0,2025.0,1.0,41051404,True,2025.0,test,3,train,test
4,WPQIAQFAPSASAFFGMS,228.0,0.0,18,0.00000,1,0,2025.0,2025.0,1.0,41051404,True,2025.0,test,4,train,test


In [3]:
summary_path = PROJECT/'02_data/splits/split_summary_v1.json'
assert summary_path.exists()
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2)[:8000])
rows=[]
for design,col in [('exact','exact_split'),('cluster','cluster_split'),('temporal','temporal_split')]:
    for partition in ['train','test']:
        q=splits[splits[col].eq(partition)]
        rows.append({'split':design,'partition':partition,'n_peptides':len(q),'positive_rate':q.label_hiconf.mean(),
                     'earliest_year_min':q.earliest_pub_year.min(),'earliest_year_max':q.earliest_pub_year.max()})
tidy=pd.DataFrame(rows)
tidy.to_csv(TABLES/'table_01_split_summary.csv', index=False)
display(tidy)

{
  "cluster": {
    "excluded": {
      "clusters": 0,
      "n": 4,
      "positive_n": 0,
      "positive_rate": 0.0,
      "unique_peptides": 4
    },
    "test": {
      "clusters": 1658,
      "n": 1933,
      "positive_n": 178,
      "positive_rate": 0.09208484221417486,
      "unique_peptides": 1933
    },
    "train": {
      "clusters": 6628,
      "n": 7735,
      "positive_n": 712,
      "positive_rate": 0.092049127343245,
      "unique_peptides": 7735
    }
  },
  "cluster_count": 8286,
  "cluster_definition": "transitive connected components at Levenshtein edit distance <= 2",
  "cluster_overlap_train_test": 0,
  "cluster_split_search_seeds": 512,
  "cluster_split_selected_seed": 22,
  "exact": {
    "excluded": {
      "clusters": 0,
      "n": 4,
      "positive_n": 0,
      "positive_rate": 0.0,
      "unique_peptides": 4
    },
    "test": {
      "clusters": 1819,
      "n": 1934,
      "positive_n": 178,
      "positive_rate": 0.09203722854188211,
      "unique_pept

,split,partition,n_peptides,positive_rate,earliest_year_min,earliest_year_max
0,exact,train,7734,0.092061,1996.0,2025.0
1,exact,test,1934,0.092037,2000.0,2025.0
2,cluster,train,7735,0.092049,1996.0,2025.0
3,cluster,test,1933,0.092085,2001.0,2025.0
4,temporal,train,7732,0.086006,1996.0,2020.0
5,temporal,test,1935,0.116279,2021.0,2025.0


**Interpretation.** Exact peptide-disjoint evaluation is the least stringent; edit-distance ≤2 component splitting addresses near-neighbor leakage; temporal testing evaluates later records but is not simultaneously cluster-purged.